# edge-microdot DuckDB sweep (PC)

Copy `edge_history.duckdb` from a Pi (`scp`) and point `DB_PATH` at the read-only copy. Runs the same SQL rules as `examples/edge-microdot/sql/`.

In [ ]:
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt

DB_PATH = Path("../var/edge_history.duckdb")
SQL_DIR = Path("../sql")
assert DB_PATH.is_file(), "Copy the Pi DuckDB file first"
con = duckdb.connect(str(DB_PATH), read_only=True)

In [ ]:
stale_sql = (SQL_DIR / "01_stale.sql").read_text()
stale = con.execute(stale_sql, [10, 10]).df()
display(stale.head(20))

In [ ]:
ts = con.execute(
    """
    SELECT date_trunc('minute', timestamp_utc) AS bucket,
           avg(value_num) AS avg_c
    FROM telemetry
    WHERE value_num IS NOT NULL
    GROUP BY 1 ORDER BY 1
    """
).df()
plt.figure(figsize=(10, 3))
plt.plot(ts["bucket"], ts["avg_c"])
plt.title("Site average (numeric points)")
plt.tight_layout()
plt.show()